# SvasaLens: export the networks for the XIAO
Makes `svweights.h` (the trained two-hospital detection network + the NN v2 typing network) for the V2 firmware.

Runtime → **Run all**. No GPU needed. It takes about a minute, then `svweights.h` downloads. Put it in the `svasalens_v2` sketch folder, replacing the one there.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import glob, os
DRIVE = '/content/drive/MyDrive'
def find(name, prefer):
    p = f'{DRIVE}/{prefer}/{name}'
    if os.path.exists(p): return p
    hits = glob.glob(f'{DRIVE}/**/{name}', recursive=True)
    assert hits, f'{name} not found anywhere in My Drive'
    return hits[0]
DET = find('det_two_hospitals.pt', 'SvasaLens_Detection_2H')
TYP = find('NN_v2_effort_two_hospitals.pt', 'SvasaLens_NN_v2')
print('detection network:', DET); print('typing network:   ', TYP)

In [ ]:
EXPORT_PY = "\"\"\"Turn the two trained SvasaLens networks into svweights.h for the XIAO firmware.\n\nusage: python export_weights.py DET_MODEL.pt TYPING_MODEL.pt OUT_svweights.h\n\nThe header also carries a self-test: a made-up input and the answers PyTorch gives for it.\nThe firmware runs the same input at start-up and prints SELF-TEST PASS if the chip agrees.\"\"\"\nimport sys, time\nimport numpy as np\nimport torch, torch.nn as nn\n\n\n# ---- the two networks, exactly as trained (svdetnn.DetNet and the NN v2 Net)\nclass CausalConv(nn.Module):\n    def __init__(s, cin, cout, k, d):\n        super().__init__(); s.pad = (k - 1) * d; s.c = nn.Conv1d(cin, cout, k, dilation=d)\n    def forward(s, x): return s.c(nn.functional.pad(x, (s.pad, 0)))\n\n\nclass DetNet(nn.Module):\n    def __init__(s, cin, ch=16, k=9, dil=(1, 2, 4, 8, 16, 32, 64)):\n        super().__init__()\n        layers, c = [], cin\n        for d in dil:\n            layers += [CausalConv(c, ch, k, d), nn.ReLU()]; c = ch\n        s.body = nn.Sequential(*layers); s.head = nn.Conv1d(ch, 1, 1)\n    def forward(s, x): return s.head(s.body(x))[:, 0]          # logit for every sample\n\n\nclass TypeNet(nn.Module):\n    def __init__(s, ch):\n        super().__init__()\n        s.f = nn.Sequential(nn.Conv1d(ch, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),\n                            nn.Conv1d(16, 32, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),\n                            nn.Conv1d(32, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1))\n        s.d = nn.Dropout(0.3); s.o = nn.Linear(32, 1)\n    def forward(s, x): return s.o(s.d(s.f(x).squeeze(-1))).squeeze(-1)\n\n\ndef arr(name, a, fmt='%.9g'):\n    a = np.asarray(a, np.float32)\n    dims = ''.join(f'[{d}]' for d in a.shape)\n    body = ', '.join(fmt % v for v in a.ravel())\n    return f'static const float {name}{dims} = {{{body}}};\\n'\n\n\ndef main(det_f, typ_f, out_f):\n    ck = torch.load(det_f, map_location='cpu', weights_only=False)\n    ct = torch.load(typ_f, map_location='cpu', weights_only=False)\n    dch, alarm = list(ck['channels']), ck['alarm']\n    tch, cut = list(ct['channels']), float(ct['cut'])\n    det = DetNet(len(dch)); det.load_state_dict(ck['weights']); det.eval()\n    typ = TypeNet(len(tch)); typ.load_state_dict(ct['weights']); typ.eval()\n    w = {k: v.numpy() for k, v in ck['weights'].items()}\n    t = {k: v.numpy() for k, v in ct['weights'].items()}\n\n    # self-test input: 200 s of breathing-like waves with pauses, plus one typing window\n    rng = np.random.default_rng(1)\n    T = 200 * 8\n    tt = np.arange(T) / 8\n    xin = np.stack([np.sin(2 * np.pi * (0.25 + 0.03 * c) * tt) * (1 + 0.5 * np.sin(tt / 23 + c))\n                    + 0.2 * rng.normal(size=T) for c in range(len(dch))]).astype(np.float32)\n    xin[:, 60 * 8:80 * 8] *= 0.05                                             # a pause\n    with torch.no_grad():\n        logit = det(torch.tensor(xin[None]))[0].numpy()[7::8]                  # one per second\n        win = rng.normal(size=(len(tch), 720)).astype(np.float32)\n        win[-1] = 0; win[-1, 240:400] = 1                                     # last row: event marker\n        score = float(typ(torch.tensor(win[None]))[0])\n\n    L = [f'// SvasaLens network weights, made by export_weights.py on {time.strftime(\"%Y-%m-%d %H:%M\")}\\n',\n         f'// detection: {det_f}\\n// typing:    {typ_f}\\n#pragma once\\n\\n',\n         f'static const int DET_CIN = {len(dch)};\\n',\n         f'static const int DET_CH[{len(dch)}] = {{{\", \".join(map(str, dch))}}};   // 0 thermistor, 1 chest, 2 belly\\n',\n         f'static const float ALARM_THR = {float(alarm[\"thr\"]):.6f}f;\\n',\n         f'static const int ALARM_MIN_DUR = {int(alarm[\"min_dur\"])};\\n',\n         f'static const int ALARM_LAG = {int(alarm[\"lag\"])};\\n',\n         arr('DET_W0', w['body.0.c.weight']), arr('DET_B0', w['body.0.c.bias']),\n         arr('DET_W', np.stack([w[f'body.{2 * i}.c.weight'] for i in range(1, 7)])),\n         arr('DET_B', np.stack([w[f'body.{2 * i}.c.bias'] for i in range(1, 7)])),\n         arr('DET_HW', w['head.weight'][0, :, 0]),\n         f'static const float DET_HB = {float(w[\"head.bias\"][0]):.9g}f;\\n\\n',\n         f'static const int TYP_CIN = {len(tch)};\\n',\n         f'static const int TYP_CH[{len(tch)}] = {{{\", \".join(map(str, tch))}}};   // 0 chest, 1 belly, 2 airflow, 3 event marker\\n',\n         f'static const float TYP_CUT = {cut:.9g}f;\\n',\n         arr('TYP_W1', t['f.0.weight']), arr('TYP_B1', t['f.0.bias']),\n         arr('TYP_W2', t['f.3.weight']), arr('TYP_B2', t['f.3.bias']),\n         arr('TYP_W3', t['f.6.weight']), arr('TYP_B3', t['f.6.bias']),\n         arr('TYP_OW', t['o.weight'][0]), f'static const float TYP_OB = {float(t[\"o.bias\"][0]):.9g}f;\\n\\n',\n         '// self-test (input and PyTorch answers)\\n',\n         f'#define SELFTEST 1\\nstatic const int ST_SEC = 200;\\n',\n         arr('ST_DET_IN', xin.T), arr('ST_DET_LOGIT', logit),\n         arr('ST_TYP_IN', win), f'static const float ST_TYP_SCORE = {score:.9g}f;\\n']\n    open(out_f, 'w').write(''.join(L))\n    print(f'wrote {out_f}: alarm {alarm}, detection channels {dch}, typing channels {tch}, cut {cut:.4f}')\n\n\nif __name__ == '__main__':\n    main(*sys.argv[1:4])\n"
open('/content/export_weights.py', 'w').write(EXPORT_PY)

In [ ]:
import sys; sys.path.insert(0, '/content')
import export_weights as E
OUT = f'{DRIVE}/SvasaLens_Firmware'; os.makedirs(OUT, exist_ok=True)
E.main(DET, TYP, f'{OUT}/svweights.h')
print('saved to', f'{OUT}/svweights.h', f'({os.path.getsize(OUT + "/svweights.h") // 1024} KB)')
files.download(f'{OUT}/svweights.h')